# CinderTrace — Sri Lanka 2023 authoritative-source check

This notebook checks the **existing immutable pilot**, without changing the frontend or rebuilding historical analysis. It retrieves the same three standard-product streams directly from NASA FIRMS and reconciles all 2023 observations inside the exact supplied polygon.

This is an independent retrieval check of the mirror against its upstream source. It is **not independent fire truth**, overpass/cloud validation, cross-sensor calibration or evidence that a recorded zero means no fire.

## Before running

1. Have `cindertrace_atlas_lk_2023_pilot.zip` available. Its expected SHA256 is checked below.
2. Obtain a free **MAP_KEY** from [NASA FIRMS API](https://firms.modaps.eosdis.nasa.gov/api/area/). Use its **Get MAP_KEY** control. Do not paste the key into chat or notebook source.
3. Run cells in order. Enter the key only at the hidden prompt. No Earthdata password is needed.
4. The default checks the entire year: 219 small requests, no more than five days per request, across MODIS, S-NPP and NOAA-20. Verified downloads are cached; reruns reuse them. A failed request stops the check and never becomes zero.
5. Save the executed notebook and the final results ZIP. Discrepancies remain review items, not automatic edits.

The four-satellite scope matches this mirror pilot. NOAA-21 is not included; NASA and the mirror describe different historical availability for it. Do not call this a complete five-satellite NASA inventory.


In [20]:
%pip -q install pandas pyarrow shapely


## 1. Choose local files or Google Drive
Colab mounts Drive for durable output. VS Code uses the project folder and never calls the browser-only upload function. Change `BUNDLE_PATH` if needed.


In [27]:
from pathlib import Path
from datetime import datetime,timezone,timedelta
from io import BytesIO
import hashlib,json,zipfile,sys,time
import pandas as pd
import numpy as np
import shapely
from IPython.display import display

IN_COLAB='google.colab' in sys.modules
BUNDLE_NAME='cindertrace_atlas_lk_2023_pilot.zip'
BUNDLE_PATH=None  # Optional: Path('/your/path/cindertrace_atlas_lk_2023_pilot.zip')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT=Path('/content/drive/MyDrive/cindertrace_atlas/verification/lk_2023_official')
else:
    OUTPUT_ROOT=Path.cwd()/'atlas_lk_2023_official'
if BUNDLE_PATH is None:
    candidates=[Path.cwd()/BUNDLE_NAME,Path.cwd().parent/BUNDLE_NAME,Path('/content')/BUNDLE_NAME]
    if IN_COLAB:
        candidates+=list(Path('/content/drive/MyDrive/cindertrace_atlas/runs').glob('*/'+BUNDLE_NAME))
    BUNDLE_PATH=next((p for p in candidates if p.is_file()),None)
if BUNDLE_PATH is None and IN_COLAB:
    from google.colab import files
    uploaded=files.upload()
    assert BUNDLE_NAME in uploaded,'Select the original pilot ZIP.'
    BUNDLE_PATH=Path('/content')/BUNDLE_NAME
    BUNDLE_PATH.write_bytes(uploaded[BUNDLE_NAME])
if BUNDLE_PATH is None or not Path(BUNDLE_PATH).is_file():
    raise FileNotFoundError('Set BUNDLE_PATH to the pilot ZIP and rerun this cell. VS Code does not support Colab files.upload().')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print('Input:',Path(BUNDLE_PATH).name)
print('Results folder:',OUTPUT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input: cindertrace_atlas_lk_2023_pilot.zip
Results folder: /content/drive/MyDrive/cindertrace_atlas/verification/lk_2023_official


## 2. Verify the original pilot and exact boundary
The ZIP is read in memory; archive paths are never extracted. The bundle and all inner hashes must match.


In [28]:
EXPECTED_SHA256='11184caea4d9ca669e4f7f455d120d8f4d898db0bf2086f5c86cb9b47650e49e'
EXPECTED_MEMBERS={'audit_manifest.json','observations.parquet','monthly_observed_counts.csv','sensor_counts.csv','region.geojson','source_schema.csv'}
raw=Path(BUNDLE_PATH).read_bytes()
assert hashlib.sha256(raw).hexdigest()==EXPECTED_SHA256,'Wrong or changed pilot ZIP. Stop and review.'
with zipfile.ZipFile(BytesIO(raw)) as archive:
    assert len(archive.namelist())==6 and set(archive.namelist())==EXPECTED_MEMBERS
    assert all(i.file_size<30_000_000 for i in archive.infolist())
    supplied={name:archive.read(name) for name in archive.namelist()}
AUDIT=json.loads(supplied['audit_manifest.json'])
assert set(AUDIT['files'])==EXPECTED_MEMBERS-{'audit_manifest.json'}
for name,receipt in AUDIT['files'].items():
    assert len(supplied[name])==receipt['bytes'] and hashlib.sha256(supplied[name]).hexdigest()==receipt['sha256'],name
assert AUDIT['source_before']==AUDIT['source_after']
MIRROR=pd.read_parquet(BytesIO(supplied['observations.parquet']))
REGION=shapely.geometry.shape(json.loads(supplied['region.geojson'])['features'][0]['geometry'])
assert REGION.is_valid and len(MIRROR)==13366 and not MIRROR.duplicated().any()
assert MIRROR.quality.eq('sp').all()
assert shapely.covers(REGION,shapely.from_wkb(MIRROR.geometry_wkb.map(bytes).to_numpy())).all()
MIRROR['acq_datetime']=pd.to_datetime(MIRROR.acq_datetime,utc=True)
assert MIRROR.acq_datetime.dt.year.eq(2023).all()
assert MIRROR.groupby('satellite').size().to_dict()=={'Aqua':642,'NOAA-20':6094,'Suomi-NPP':6432,'Terra':198}
BBOX=tuple(float(x) for x in AUDIT['query_parameters'][3:7])
x0,y0,x1,y1=REGION.bounds
assert BBOX[0]<=x0 and BBOX[1]<=y0 and BBOX[2]>=x1 and BBOX[3]>=y1,'Query bbox does not enclose boundary.'
print('Verified original pilot:',len(MIRROR),'records. Same polygon and bbox will be used against NASA.')


Verified original pilot: 13366 records. Same polygon and bbox will be used against NASA.


## 3. Load the tested comparison functions
Native MODIS percentages and VIIRS classes remain separate. Coordinate keys use five decimal degrees, with row multiplicities retained. Matching keys also have their native attributes checked; equal counts alone cannot pass.


In [29]:
"""Compare native FIRMS standard-product API rows with the checked mirror extract.
Shared by local tests and the standalone notebook. Never logs or persists MAP_KEY.
"""
from io import BytesIO
from collections import Counter
import json
import numpy as np
import pandas as pd
import shapely

SOURCES={'MODIS_SP':{'T':'Terra','A':'Aqua','Terra':'Terra','Aqua':'Aqua'},
 'VIIRS_SNPP_SP':{'N':'Suomi-NPP','Suomi-NPP':'Suomi-NPP'},
 'VIIRS_NOAA20_SP':{'1':'NOAA-20','NOAA-20':'NOAA-20'}}


def parse_official_csv(raw,source,start,end,boundary):
    """Reject failed/error/changed-schema responses; clip a bounding-box result exactly."""
    if source not in SOURCES:raise ValueError('Unsupported standard-product source')
    try:frame=pd.read_csv(BytesIO(raw),dtype={'acq_time':'string','satellite':'string','confidence':'string','version':'string'})
    except Exception:raise ValueError('FIRMS response is not valid CSV; do not interpret it as zero') from None
    base={'latitude','longitude','acq_date','acq_time','satellite','instrument','confidence','version','type','frp','scan','track','daynight'}
    required=base|({'brightness','bright_t31'} if source=='MODIS_SP' else {'bright_ti4','bright_ti5'})
    if not required.issubset(frame):raise ValueError('FIRMS CSV schema is incomplete or changed; response is not an observed zero')
    satellite=frame.satellite.map(SOURCES[source])
    if satellite.isna().any():raise ValueError('Unexpected satellite code; review source mapping')
    times=frame.acq_time.astype('string').str.zfill(4)
    if not times.str.match(r'^\d{4}$').all():raise ValueError('Invalid acquisition time')
    try:
        acquired=pd.to_datetime(frame.acq_date.astype(str)+' '+times.str[:2]+':'+times.str[2:],utc=True,errors='raise')
        lon=pd.to_numeric(frame.longitude,errors='raise');lat=pd.to_numeric(frame.latitude,errors='raise')
    except Exception:raise ValueError('Invalid native date/time/coordinate fields') from None
    if not np.isfinite(lon).all() or not np.isfinite(lat).all() or not lon.between(-180,180).all() or not lat.between(-90,90).all():raise ValueError('Invalid coordinates')
    if not acquired.ge(pd.Timestamp(start,tz='UTC')).all() or not acquired.lt(pd.Timestamp(end,tz='UTC')).all():raise ValueError('API returned observations outside the requested window')
    frame['satellite']=satellite;frame['acq_datetime']=acquired;frame['audit_lon']=lon;frame['audit_lat']=lat
    if frame.version.str.contains('NRT',case=False,na=True).any():raise ValueError('NRT rows received for a standard-product query')
    if not frame.daynight.isin(['D','N']).all():raise ValueError('Invalid day/night value')
    if not frame.type.isin([0,1,2,3]).all():raise ValueError('Missing or unsupported standard-product type')
    if source=='MODIS_SP':
        numeric=pd.to_numeric(frame.confidence,errors='raise')
        if not (numeric.between(0,100)&numeric.eq(numeric.round())).all():raise ValueError('Invalid native MODIS confidence')
        frame['confidence']=numeric.map(lambda v:str(int(v)))
        frame['confidence_pct']=numeric
    else:
        if not frame.confidence.isin(['l','n','h']).all():raise ValueError('Invalid native VIIRS confidence')
        frame['confidence_pct']=np.nan
    for column in ['frp','scan','track']:
        numeric=pd.to_numeric(frame[column],errors='raise')
        if not np.isfinite(numeric).all() or numeric.lt(0).any():raise ValueError('Invalid native numeric field')
        frame[column]=numeric
    inside=shapely.covers(boundary,shapely.points(lon.to_numpy(),lat.to_numpy()))
    return frame.loc[inside].copy()


def native_keys(frame):
    """Published coordinates compared at 5 decimal degrees; multiplicities preserved."""
    times=pd.to_datetime(frame.acq_datetime,utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    return pd.Series([f'{s}|{t}|{float(x):.5f}|{float(y):.5f}' for s,t,x,y in zip(frame.satellite,times,frame.audit_lon,frame.audit_lat)],index=frame.index)


def compare_native(mirror,official):
    left=mirror.copy();right=official.copy();left['record_key']=native_keys(left);right['record_key']=native_keys(right)
    lc,rc=Counter(left.record_key),Counter(right.record_key)
    missing=[{'record_key':k,'rows':n} for k,n in (rc-lc).items()]
    extra=[{'record_key':k,'rows':n} for k,n in (lc-rc).items()]
    duplicate_keys=any(n>1 for n in lc.values()) or any(n>1 for n in rc.values())
    differences=[]
    if not duplicate_keys:
        l=left.set_index('record_key');r=right.set_index('record_key')
        columns=['confidence','type','daynight','instrument','version','frp','scan','track','brightness','bright_t31','bright_ti4','bright_ti5']
        common=l.index.intersection(r.index)
        numeric={'type','frp','scan','track','brightness','bright_t31','bright_ti4','bright_ti5'}
        for col in columns:
            if col not in l or col not in r:continue # sensor-inapplicable channels remain null / absent.
            a,b=l.loc[common,col],r.loc[common,col]
            if col in numeric:
                av=pd.to_numeric(a,errors='raise').to_numpy(dtype=float,na_value=np.nan);bv=pd.to_numeric(b,errors='raise').to_numpy(dtype=float,na_value=np.nan)
                same=np.isclose(av,bv,atol=1e-5,rtol=0,equal_nan=True)
            else:same=(a.astype('string').fillna('<NULL>')==b.astype('string').fillna('<NULL>')).to_numpy(dtype=bool)
            for key in common[~same]:differences.append({'record_key':key,'field':col,'mirror':str(l.loc[key,col]),'official':str(r.loc[key,col])})
    status='matched_native_records' if not (missing or extra or differences or duplicate_keys) else 'review_required'
    return {'status':status,'mirror_rows':len(left),'official_rows':len(right),'shared_rows':sum((lc&rc).values()),
      'official_rows_missing_in_mirror':sum((rc-lc).values()),'mirror_rows_not_in_official':sum((lc-rc).values()),
      'duplicate_keys_need_review':duplicate_keys,'native_attribute_differences':len(differences),
      'coordinate_comparison_decimals':5,'numeric_comparison_abs_tolerance':1e-5,
      'coverage_verified':False,'missing_in_mirror':missing,'extra_in_mirror':extra,'attribute_differences':differences}


## 4. Enter your FIRMS MAP_KEY privately
The key stays in runtime memory. It is never written into result files, receipts or displayed request URLs. Get a new key if it has previously appeared in saved notebook source/output.


In [30]:
from getpass import getpass
MAP_KEY=getpass('NASA FIRMS MAP_KEY (hidden): ').strip()
assert MAP_KEY and MAP_KEY.replace('-','').isalnum(),'Enter the MAP_KEY, not an Earthdata password.'
print('Key entered privately. It will not be included in exported results.')


Key entered privately. It will not be included in exported results.


In [33]:
SOURCES['VIIRS_NOAA20_SP']['N20'] = 'NOAA-20'

print('Updated NOAA-20 mapping:', SOURCES['VIIRS_NOAA20_SP'])

Updated NOAA-20 mapping: {'1': 'NOAA-20', 'NOAA-20': 'NOAA-20', 'N20': 'NOAA-20'}


## 5. Retrieve the full 2023 comparison cohort
Only explicit standard-product requests are made. Cached CSVs need matching request parameters and a valid checksum. HTTP failures, invalid CSVs and changed schemas stop the run. Native records are clipped to the identical polygon.


In [34]:
from urllib.request import Request,urlopen
from urllib.error import HTTPError,URLError

CACHE=OUTPUT_ROOT/'cache';CACHE.mkdir(exist_ok=True)
windows=[]
for source in SOURCES:
    start=pd.Timestamp('2023-01-01')
    while start<pd.Timestamp('2024-01-01'):
        end=min(start+pd.Timedelta(days=5),pd.Timestamp('2024-01-01'))
        windows.append((source,start.strftime('%Y-%m-%d'),end.strftime('%Y-%m-%d')))
        start=end
assert len(windows)==219

def retrieve(source,start,end):
    request={'source':source,'start_inclusive_utc':start,'end_exclusive_utc':end,'bbox':list(BBOX)}
    stem=f'{source}_{start}_{end}'
    csv_path=CACHE/(stem+'.csv');receipt_path=CACHE/(stem+'.receipt.json')
    if csv_path.exists() and receipt_path.exists():
        payload=csv_path.read_bytes();receipt=json.loads(receipt_path.read_text())
        assert receipt['request']==request,'Cached request differs. Keep the old files for review and use a fresh output folder.'
        assert hashlib.sha256(payload).hexdigest()==receipt['sha256'],'Cached CSV changed. Stop and review.'
        return parse_official_csv(payload,source,start,end,REGION),receipt
    days=(pd.Timestamp(end)-pd.Timestamp(start)).days
    # Construct the credential-bearing URL only in memory; never log it or serialize it.
    url='https://firms.modaps.eosdis.nasa.gov/api/area/csv/'+MAP_KEY+'/'+source+'/'+','.join(map(str,BBOX))+'/'+str(days)+'/'+start
    payload=None;status='network'
    for attempt in range(4):
        try:
            with urlopen(Request(url,headers={'User-Agent':'CinderTrace-2023-source-verification/1.0'}),timeout=90) as response:
                payload=response.read()
            break
        except HTTPError as error:
            status='HTTP '+str(error.code)
            if error.code not in [429,500,502,503,504]:break
        except (URLError,TimeoutError,OSError):status='network/timeout'
        if attempt<3:time.sleep(min(2**(attempt+1),8))
    if payload is None:
        raise RuntimeError(f'NASA request failed ({status}) for {source} {start}. No zero count was written. Check the key/service and rerun; verified cached files are retained.') from None
    frame=parse_official_csv(payload,source,start,end,REGION)
    receipt={'request':request,'retrieved_utc':datetime.now(timezone.utc).isoformat(),'bytes':len(payload),
             'sha256':hashlib.sha256(payload).hexdigest(),'regional_rows':len(frame),'request_status':'csv_schema_and_scope_verified'}
    csv_path.write_bytes(payload)
    receipt_path.write_text(json.dumps(receipt,indent=2))
    return frame,receipt

OFFICIAL_PARTS=[];RECEIPTS=[]
for index,(source,start,end) in enumerate(windows,1):
    frame,receipt=retrieve(source,start,end)
    OFFICIAL_PARTS.append(frame);RECEIPTS.append(receipt)
    if index%15==0 or index==len(windows):print(f'Checked {index}/{len(windows)} requests; {sum(len(f) for f in OFFICIAL_PARTS):,} regional observations.')
    time.sleep(0.2)
OFFICIAL=pd.concat(OFFICIAL_PARTS,ignore_index=True)
assert len(RECEIPTS)==len(windows)
print('All requested windows completed. These counts are still detection observations, not unique fires or complete coverage.')


Checked 15/219 requests; 55 regional observations.
Checked 30/219 requests; 91 regional observations.
Checked 45/219 requests; 500 regional observations.
Checked 60/219 requests; 836 regional observations.
Checked 75/219 requests; 856 regional observations.
Checked 90/219 requests; 1,189 regional observations.
Checked 105/219 requests; 1,651 regional observations.
Checked 120/219 requests; 5,775 regional observations.
Checked 135/219 requests; 7,266 regional observations.
Checked 150/219 requests; 7,332 regional observations.
Checked 165/219 requests; 7,834 regional observations.
Checked 180/219 requests; 8,163 regional observations.
Checked 195/219 requests; 12,455 regional observations.
Checked 210/219 requests; 13,360 regional observations.
Checked 219/219 requests; 13,366 regional observations.
All requested windows completed. These counts are still detection observations, not unique fires or complete coverage.


/tmp/ipykernel_476/918001607.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  OFFICIAL=pd.concat(OFFICIAL_PARTS,ignore_index=True)


## 6. Compare records and native attributes
A discrepancy may be a mirror omission, source revision, version mismatch, duplicate or field change. It remains unresolved until inspected; this notebook does not overwrite the pilot.


In [35]:
COMPARISON=compare_native(MIRROR,OFFICIAL)
SUMMARY={k:v for k,v in COMPARISON.items() if k not in ['missing_in_mirror','extra_in_mirror','attribute_differences']}
display(pd.DataFrame([SUMMARY]))
for name in ['missing_in_mirror','extra_in_mirror','attribute_differences']:
    details=pd.DataFrame(COMPARISON[name]);details.to_parquet(OUTPUT_ROOT/(name+'.parquet'),index=False)
    if len(details):print(name,':',len(details),'review rows');display(details.head(10))
months=[f'2023-{m:02d}' for m in range(1,13)]
satellites=['Terra','Aqua','Suomi-NPP','NOAA-20']
expected=pd.MultiIndex.from_product([months,satellites],names=['month','satellite'])
def monthly(frame):
    return frame.assign(month=pd.to_datetime(frame.acq_datetime,utc=True).dt.strftime('%Y-%m')).groupby(['month','satellite']).size().reindex(expected,fill_value=0)
MONTHLY=pd.concat([monthly(MIRROR).rename('mirror_recorded_detections'),monthly(OFFICIAL).rename('official_recorded_detections')],axis=1).reset_index()
MONTHLY['difference']=MONTHLY.official_recorded_detections-MONTHLY.mirror_recorded_detections
MONTHLY['coverage_status']='unverified'
MONTHLY.to_parquet(OUTPUT_ROOT/'monthly_reconciliation.parquet',index=False)
OFFICIAL.to_parquet(OUTPUT_ROOT/'official_regional_observations.parquet',index=False)
display(MONTHLY)
print('Decision:',COMPARISON['status'])
print('Even a full match validates retrieval consistency, not complete satellite coverage or harmonization.')


,status,mirror_rows,official_rows,shared_rows,official_rows_missing_in_mirror,mirror_rows_not_in_official,duplicate_keys_need_review,native_attribute_differences,coordinate_comparison_decimals,numeric_comparison_abs_tolerance,coverage_verified
0,matched_native_records,13366,13366,13366,0,0,False,0,5,0.00001,False


,month,satellite,mirror_recorded_detections,official_recorded_detections,difference,coverage_status
0,2023-01,Terra,2,2,0,unverified
1,2023-01,Aqua,3,3,0,unverified
2,2023-01,Suomi-NPP,43,43,0,unverified
3,2023-01,NOAA-20,67,67,0,unverified
4,2023-02,Terra,3,3,0,unverified
5,2023-02,Aqua,13,13,0,unverified
6,2023-02,Suomi-NPP,136,136,0,unverified
7,2023-02,NOAA-20,168,168,0,unverified
8,2023-03,Terra,9,9,0,unverified
9,2023-03,Aqua,39,39,0,unverified


Decision: matched_native_records
Even a full match validates retrieval consistency, not complete satellite coverage or harmonization.


## 7. Save the checkpoint
Download this ZIP and save the executed notebook. The ZIP contains official request receipts and raw CSV responses without your key, regional observations, discrepancy rows and monthly totals. Return these for review before expanding years or fitting transfers.


In [36]:
import importlib.metadata as metadata
report={'pilot_bundle_sha256':EXPECTED_SHA256,'source_service':'https://firms.modaps.eosdis.nasa.gov/api/area/',
        'checked_sources':list(SOURCES),'scope':'Sri Lanka pilot polygon, all 2023 UTC dates, standard products, all native types/confidences',
        'boundary_sha256':AUDIT['boundary_sha256'],'completed_windows':len(RECEIPTS),'expected_windows':219,
        'comparison':COMPARISON,'coverage_verified':False,'harmonization_validated':False,
        'packages':{p:metadata.version(p) for p in ['pandas','pyarrow','shapely']},
        'request_receipts':RECEIPTS,'completed_utc':datetime.now(timezone.utc).isoformat()}
(OUTPUT_ROOT/'official_reconciliation_manifest.json').write_text(json.dumps(report,indent=2,allow_nan=False))
ZIP_PATH=OUTPUT_ROOT.parent/'cindertrace_atlas_lk_2023_official_check.zip'
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file():archive.write(path,path.relative_to(OUTPUT_ROOT))
print('Checkpoint:',ZIP_PATH)
print('SHA256:',hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
MAP_KEY=None  # Clear the notebook variable after completing retrieval.
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))


Checkpoint: /content/drive/MyDrive/cindertrace_atlas/verification/cindertrace_atlas_lk_2023_official_check.zip
SHA256: 509afdf0418a01b248671addb4d7a221a02e0e5e275fb203aa6e170206d29077


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>